# SeguWatts Vision: treino do detector

Treino customizado de uma **YOLO11 nano** para reconhecer o **estado da iluminação** de um cômodo, base da regra de desperdício do SeguWatts:

| Classe no dataset público | Vira no SeguWatts |
|---|---|
| `Lamp On`, `Light Bulb On` | `lampada_acesa` |
| `Lamp Off`, `Light Bulb Off` | `lampada_apagada` |

As **pessoas** são detectadas pela YOLO pré-treinada no COCO, que o detector roda junto no mesmo quadro. A tradução dos nomes fica em `seguwatts_vision/config.py` (`normalizar_classe`).

**Antes de rodar**

1. *Ambiente de execução → Alterar o tipo → GPU T4*.
2. Crie uma conta gratuita em roboflow.com e copie a chave em *Settings → API Keys*.
3. Em 🔑 *Secrets* (barra lateral do Colab), crie `ROBOFLOW_API_KEY` com a chave e libere o acesso deste notebook. A chave nunca fica escrita no notebook.

Tempo total: cerca de 30 a 40 minutos.

In [ ]:
# 1. Dependências (mesma versão do requirements.txt do repositório)
!pip install -q ultralytics==8.4.214 roboflow
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())

In [ ]:
# 2. Dataset público do Roboflow Universe (já anotado)
from google.colab import userdata
from roboflow import Roboflow

# Dataset público de lâmpadas acesas/apagadas. Para usar outro, troque as duas linhas
# (o endereço do Universe é universe.roboflow.com/<WORKSPACE>/<PROJETO>).
WORKSPACE = "infocomm-thingy"
PROJETO = "light-detection-gecjq"

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
projeto = rf.workspace(WORKSPACE).project(PROJETO)
versoes = sorted(int(str(v.version).split("/")[-1]) for v in projeto.versions())
print("versões disponíveis:", versoes, "→ usando a mais recente")
dataset = projeto.version(versoes[-1]).download("yolov8")
DATA_YAML = f"{dataset.location}/data.yaml"
print(open(DATA_YAML).read())

# Anote no relatório: link do dataset, versão, número de imagens e licença
# (aparecem na página do dataset no Universe e no README.roboflow.txt baixado).
!cat "{dataset.location}/README.roboflow.txt" 2>/dev/null | head -20

# Alternativa: dataset próprio exportado do Roboflow ou um .zip no Drive
# from google.colab import drive; drive.mount('/content/drive')
# !unzip -q /content/drive/MyDrive/seguwatts-vision.zip -d /content/dataset
# DATA_YAML = "/content/dataset/data.yaml"

In [ ]:
# 3. Quantas instâncias de cada classe existem em cada divisão (vai para o relatório)
import collections, glob, os, yaml, random, shutil
import pandas as pd

info = yaml.safe_load(open(DATA_YAML))
nomes = info["names"]
nomes = nomes if isinstance(nomes, list) else [nomes[i] for i in sorted(nomes)]
base = os.path.dirname(DATA_YAML)

# Alguns datasets públicos não têm divisão de teste. Se faltar, separamos 20% da
# validação como teste, para avaliar o modelo em imagens que ele nunca viu.
if not glob.glob(f"{base}/test/images/*"):
    imgs = sorted(glob.glob(f"{base}/valid/images/*"))
    random.Random(42).shuffle(imgs)
    os.makedirs(f"{base}/test/images", exist_ok=True); os.makedirs(f"{base}/test/labels", exist_ok=True)
    for img in imgs[: max(1, len(imgs) // 5)]:
        rot = img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        shutil.move(img, f"{base}/test/images/")
        if os.path.exists(rot):
            shutil.move(rot, f"{base}/test/labels/")
    info["test"] = "../test/images"
    yaml.safe_dump(info, open(DATA_YAML, "w"))
    print("divisão de teste criada a partir da validação")

linhas = []
for divisao in ["train", "valid", "test"]:
    contagem = collections.Counter()
    arquivos = glob.glob(f"{base}/{divisao}/labels/*.txt")
    for arq in arquivos:
        for linha in open(arq):
            if linha.strip():
                contagem[nomes[int(linha.split()[0])]] += 1
    linhas.append({"divisão": divisao, "imagens": len(arquivos), **{n: contagem[n] for n in nomes}})
tabela = pd.DataFrame(linhas).set_index("divisão")
tabela

In [ ]:
# 4. Treino: transfer learning a partir dos pesos COCO
from ultralytics import YOLO

# project com caminho absoluto: com caminho relativo, o Ultralytics 8.4 salva em
# runs/detect/runs/... e as células seguintes não encontram os arquivos.
modelo = YOLO("yolo11n.pt")
modelo.train(
    data=DATA_YAML,
    epochs=80,
    patience=20,       # para cedo se a validação parar de melhorar
    imgsz=640,
    batch=16,
    seed=42,           # reprodutível
    project="/content/runs",
    name="seguwatts",
    exist_ok=True,
)

In [ ]:
# 5. Avaliação no conjunto de TESTE (imagens que o modelo nunca viu no treino)
import json

melhor = YOLO("runs/seguwatts/weights/best.pt")
met = melhor.val(data=DATA_YAML, split="test", imgsz=640, plots=True, project="/content/runs", name="teste", exist_ok=True)

por_classe = {
    nomes[c]: {
        "precisao": round(float(met.box.p[i]), 3),
        "recall": round(float(met.box.r[i]), 3),
        "mAP50": round(float(met.box.ap50[i]), 3),
        "mAP50_95": round(float(met.box.ap[i]), 3),
    }
    for i, c in enumerate(met.box.ap_class_index)
}
resumo = {
    "mAP50": round(float(met.box.map50), 3),
    "mAP50_95": round(float(met.box.map), 3),
    "precisao": round(float(met.box.mp), 3),
    "recall": round(float(met.box.mr), 3),
    "por_classe": por_classe,
}
os.makedirs("resultados", exist_ok=True)
json.dump(resumo, open("resultados/metricas_teste.json", "w"), indent=2, ensure_ascii=False)
pd.DataFrame(por_classe).T

In [ ]:
# 6. Curvas de treino, matriz de confusão e curva precisão x recall
from IPython.display import Image, display
for arquivo in ["runs/seguwatts/results.png",
                "runs/teste/confusion_matrix_normalized.png",
                "runs/teste/BoxPR_curve.png"]:
    if os.path.exists(arquivo):
        print(arquivo); display(Image(arquivo, width=900))

In [ ]:
# 7. Exemplos de detecção em imagens de teste
import random, shutil
amostras = random.Random(42).sample(glob.glob(f"{base}/test/images/*"), k=min(8, len(glob.glob(f"{base}/test/images/*"))))
melhor.predict(amostras, imgsz=640, conf=0.5, save=True, project="/content/runs", name="exemplos", exist_ok=True)
for img in sorted(glob.glob("runs/exemplos/*"))[:4]:
    display(Image(img, width=600))

In [ ]:
# 8. FPS em CPU (simula o notebook do morador, sem GPU)
import time, cv2, numpy as np
quadros = [cv2.imread(a) for a in amostras]
for imgsz in (640, 480):
    melhor.predict(quadros[0], imgsz=imgsz, device="cpu", verbose=False)  # aquecimento
    t = time.perf_counter()
    n = 0
    for _ in range(5):
        for q in quadros:
            melhor.predict(q, imgsz=imgsz, device="cpu", verbose=False); n += 1
    print(f"imgsz={imgsz}: {n / (time.perf_counter() - t):.1f} FPS em CPU (Colab)")

In [ ]:
# 9. Empacota pesos e resultados para o repositório
for origem, destino in [
    ("runs/seguwatts/weights/best.pt", "modelo/best.pt"),
    ("runs/seguwatts/results.png", "resultados/curvas_treino.png"),
    ("runs/teste/confusion_matrix_normalized.png", "resultados/matriz_confusao.png"),
    ("runs/teste/BoxPR_curve.png", "resultados/curva_pr.png"),
]:
    if os.path.exists(origem):
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        shutil.copy(origem, destino)
for i, img in enumerate(sorted(glob.glob("runs/exemplos/*"))[:6]):
    shutil.copy(img, f"resultados/exemplo_{i + 1}{os.path.splitext(img)[1]}")
tabela.to_csv("resultados/contagem_classes.csv")
!zip -qr seguwatts_vision_resultados.zip modelo resultados
from google.colab import files
files.download("seguwatts_vision_resultados.zip")

## Demonstração com um vídeo do celular (sem webcam e sem instalar nada)

Grave no celular, **na horizontal e com o celular parado** (apoiado em um móvel), um vídeo de **40 a 60 segundos** no mesmo cômodo das fotos:

1. **0–10 s:** luz (ou TV) ligada **com uma pessoa** no cômodo → o painel mostra *ocupado*
2. **10–35 s:** a pessoa **sai** e o aparelho continua ligado → contador e, depois de 10 s, a faixa **"Possível desperdício"**
3. **35–50 s:** a pessoa **volta** → o alerta encerra

A célula abaixo roda o detector real do repositório (`seguwatts_vision/detector.py`) quadro a quadro, com o modelo que você acabou de treinar, e devolve o vídeo anotado e o CSV de eventos.

In [ ]:
# 10. Processa o vídeo do celular com o detector do SeguWatts Vision
import os, subprocess, pandas as pd
from google.colab import files

if not os.path.exists("repo"):
    !git clone -q https://github.com/dejesuscaua/Gs-visaocomputacional.git repo
print("Selecione o vídeo gravado no celular...")
enviado = files.upload()
VIDEO = os.path.abspath(next(iter(enviado)))
MODELO = os.path.abspath("runs/seguwatts/weights/best.pt")
os.makedirs("resultados", exist_ok=True)
SAIDA = os.path.abspath("resultados/demo_anotado.mp4")

# --demo: alerta depois de 10 s de cômodo vazio (em casa o padrão é 120 s)
!cd repo && ARQUIVO_EVENTOS=../resultados/eventos_demo.csv python -m seguwatts_vision.detector \
    --fonte "{VIDEO}" --modelo "{MODELO}" --comodo sala --demo --sem-janela --saida "{SAIDA}"

# Converte para H.264, que abre em qualquer player e no navegador
!ffmpeg -loglevel error -y -i "{SAIDA}" -vcodec libx264 -pix_fmt yuv420p -movflags +faststart resultados/demo_final.mp4
print("\nEventos gerados (vão para o webhook do n8n quando SEGUWATTS_WEBHOOK_URL está configurada):")
display(pd.read_csv("resultados/eventos_demo.csv"))
files.download("resultados/demo_final.mp4")

Se nenhum evento aparecer: confira se a sala ficou **vazia por mais de 12 s** com o aparelho ligado e se o modelo detecta o aparelho nos quadros (veja o vídeo anotado). Se a pessoa não for detectada, grave com ela mais de corpo inteiro no enquadramento.

## Ao vivo pela webcam (sem instalar nada)

A célula abaixo liga a **webcam do seu computador pelo navegador**: o Colab pede permissão de câmera, e cada quadro vai para o detector, que devolve a imagem com as caixas, o contador e a faixa **"Possível desperdício"**. São 2 a 4 quadros por segundo, porque cada quadro vai e volta pela internet. A regra mede tempo em segundos, então funciona igual.

Para a demonstração, aponte a câmera para um cômodo com uma **lâmpada acesa ou TV ligada visível**, fique no quadro, saia por mais de 12 s e volte. Para encerrar, clique no botão ⏹ da célula. Grave a tela (Win+G no Windows ou o gravador do celular) para usar no vídeo da disciplina.

In [ ]:
# 11. AO VIVO pela webcam do navegador
import base64, os, sys, time
import cv2, numpy as np, pandas as pd
from IPython.display import Javascript, Image as IPImage, display
from google.colab.output import eval_js

if not os.path.exists("repo"):
    !git clone -q https://github.com/dejesuscaua/Gs-visaocomputacional.git repo
sys.path.insert(0, os.path.abspath("repo"))
from seguwatts_vision.ao_vivo import SessaoAoVivo

MODELO = "/content/runs/seguwatts/weights/best.pt"   # modelo treinado nesta sessão
if not os.path.exists(MODELO):
    MODELO = "repo/modelo/best.pt"                     # ou o modelo publicado no repositório
DURACAO_S = 300                                        # encerra sozinho depois de 5 min
os.makedirs("/content/resultados", exist_ok=True)
sessao = SessaoAoVivo(modelo=MODELO, comodo="sala", limite_s=10, imgsz=640,
                      arquivo_eventos="/content/resultados/eventos_ao_vivo.csv")

display(Javascript("""
window.iniciarCamera = async function () {
  if (window._sw) return;
  const video = document.createElement('video');
  video.style.display = 'none';
  const stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
  video.srcObject = stream;
  await video.play();
  document.body.appendChild(video);
  const canvas = document.createElement('canvas');
  canvas.width = video.videoWidth || 640; canvas.height = video.videoHeight || 480;
  window._sw = {video, canvas, stream};
};
window.quadro = async function () {
  await window.iniciarCamera();
  const {video, canvas} = window._sw;
  canvas.getContext('2d').drawImage(video, 0, 0, canvas.width, canvas.height);
  return canvas.toDataURL('image/jpeg', 0.8);
};
window.pararCamera = function () {
  if (window._sw) { window._sw.stream.getTracks().forEach(t => t.stop()); window._sw.video.remove(); window._sw = null; }
};
"""))

tela = display(display_id=True)
inicio = time.time()
try:
    while time.time() - inicio < DURACAO_S:
        url = eval_js("quadro()")
        img = cv2.imdecode(np.frombuffer(base64.b64decode(url.split(",", 1)[1]), np.uint8), cv2.IMREAD_COLOR)
        anotado, r = sessao.processar(img)
        ok, jpg = cv2.imencode(".jpg", anotado, [cv2.IMWRITE_JPEG_QUALITY, 80])
        tela.update(IPImage(data=jpg.tobytes(), format="jpeg"))
except KeyboardInterrupt:
    print("Sessão encerrada pelo botão ⏹.")
finally:
    sessao.encerrar()
    try:
        eval_js("pararCamera()")
    except Exception:
        pass
    print(f"FPS aproximado: {sessao.fps:.1f}")
    if sessao.eventos:
        display(pd.DataFrame(sessao.eventos))
    else:
        print("Nenhum evento: a sala precisa ficar vazia por mais de ~12 s com a luz acesa.")

## Depois do treino

1. Descompacte `seguwatts_vision_resultados.zip` na raiz do repositório (pastas `modelo/` e `resultados/`) e faça commit de `modelo/best.pt` e das figuras.
2. `resultados/demo_final.mp4` é o vídeo anotado, pronto para a demonstração.
3. Copie os números de `resultados/metricas_teste.json` para o relatório (`docs/relatorio.md`).